# Carbon-X — DNA chat on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Carbon-X-inference-dna.ipynb)

[`HuggingFaceBio/Carbon-X`](https://huggingface.co/HuggingFaceBio/Carbon-X) is Hugging Face Bio's **14.8B** decoder-only LLM for DNA understanding, released on the Hub (Apache 2.0, **not gated**). It is a fine-tune of [`Qwen/Qwen3-14B`](https://huggingface.co/Qwen/Qwen3-14B), trained jointly on DNA and English. You paste **raw DNA inside an ordinary chat message** and it answers in English: benign vs pathogenic variants, regulatory-region identification, organism-group classification, next-base prediction, with a short explanation. DNA and text share one token stream; there is no separate DNA encoder.

Why this model: most genomics models are task heads or want a special DNA API. Carbon-X is a chat model. A hybrid tokenizer keeps Qwen3 BPE for text and encodes DNA as non-overlapping **6-mers** (plus tail tokens and `<dna>` / `</dna>` / `<oov>`). Runs of at least 6 A/C/G/T bases are tagged automatically. The tokenizer is **custom Hub code**, so it needs `trust_remote_code=True`. The architecture itself is stock `Qwen3ForCausalLM`. Carbon-X requires **Transformers v5+**. It was post-trained and evaluated with **thinking disabled** — keep `enable_thinking=False`.

The card reports a **66.7%** average across five DNA benchmarks with thinking off (27 points above Qwen3-14B), and it remains a capable English assistant (79.3% IFEval prompt-level strict).

This walkthrough is **inference only**. It does not train or fine-tune anything.

**Disclaimer — not clinical advice.** Outputs are **research demos**. Carbon-X has **not been clinically validated**. Do not use it to diagnose, treat, or counsel anyone, or to interpret a real patient variant. Genetic sequence data is sensitive; this notebook runs locally in your Colab runtime.

**Licence:** Apache 2.0.

**Runtime / VRAM assumption:** *Runtime > Change runtime type > T4 GPU*. A free T4 has about **15 GB** of VRAM, **no native bfloat16 tensor cores**, and **no FlashAttention-2**. The Hub checkpoint is **bfloat16**, two shards totalling about **30 GB** (`model-00001-of-00002.safetensors` 25.0 GB + `model-00002-of-00002.safetensors` 4.7 GB). That will not fit. This notebook therefore loads **4-bit NF4** with **float16 compute** via `bitsandbytes` (`low_cpu_mem_usage` streams tensors; do not load bf16 or fp16). Packed 4-bit weights are about **7.4 GB**. With a short prompt and `max_new_tokens` of 80–160, peak VRAM should land around **10–13 GB** on 15 GB — an estimate from the 4-bit size plus a small KV cache, **not a measured Colab number yet**. First run downloads ~30 GB of weights plus the pinned Qwen3 BPE files the hybrid tokenizer builds on. Attention is **SDPA** (eager if SDPA refuses the stack). Do not install `flash-attn`.

**Not in this notebook:** vLLM serving, the card's agent-tool loop / Inspect skill, GGUF (`HuggingFaceBio/Carbon-X-GGUF`), thinking mode, or generating long designed DNA.

**Links:** [model card](https://huggingface.co/HuggingFaceBio/Carbon-X) · [blog](https://huggingface.co/spaces/HuggingFaceBio/Carbon-X-Release) · [code](https://github.com/huggingface/carbon) · [GGUF](https://huggingface.co/HuggingFaceBio/Carbon-X-GGUF)



## 1. Install Transformers v5+ and bitsandbytes

Carbon-X needs **Transformers 5.13+** (the checkpoint was saved with `5.13.0.dev0`). Colab often still has a 4.x wheel imported, so this cell upgrades in place. Use `%pip` (not `python -I -m pip`) so Colab's site-packages stay on the path.

`bitsandbytes` supplies the CUDA 12 NF4 kernels used on a T4. Keep Colab's preinstalled `torch`. Do **not** install `flash-attn`.



In [ ]:
%pip install -q -U "transformers>=5.13.0,<6" "bitsandbytes>=0.46.0" "accelerate>=1.2.0" safetensors

# Carbon-X requires Transformers v5+ (card: v5 or later; saved with 5.13.0.dev0).
# bitsandbytes>=0.46 ships manylinux CUDA 12 wheels that match current Colab T4 images.
# accelerate is required for device_map="auto" / 4-bit dispatch.
# Do NOT replace Colab's torch. Do NOT install flash-attn.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.



## 2. Check the GPU and print T4 caveats

The next cell must see a CUDA GPU. It prints `torch.cuda.is_available()`, the device name, and free VRAM, then stops if you are on CPU.



In [ ]:
import gc

import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.13.0"), (
    f"transformers {transformers.__version__} is loaded; Carbon-X needs 5.13+. "
    "Run the install cell, then Runtime > Restart session."
)

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2 and no native bfloat16 tensor cores.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

DEVICE = "cuda"
major, minor = torch.cuda.get_device_capability()
NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
free, total = torch.cuda.mem_get_info()

print("transformers", transformers.__version__, "| torch", torch.__version__)
print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"VRAM free: {free / 1024**3:.1f} GB of {total / 1024**3:.1f} GB")
print("native bf16:", NATIVE_BF16, "| FlashAttention-2: off (sdpa / eager only)")
print("load path: 4-bit NF4, float16 compute (bf16 ~30 GB does not fit ~15 GB)")


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"VRAM allocated: {allocated:.2f} GB | peak: {peak:.2f} GB{suffix}")


## 3. Load Carbon-X in 4-bit NF4

`AutoTokenizer` needs `trust_remote_code=True` so `tokenizer.py` (`HybridDNATokenizer`) can run. The first tokenizer load also downloads the pinned [`Qwen/Qwen3-4B-Base`](https://huggingface.co/Qwen/Qwen3-4B-Base) BPE files the hybrid vocab is built on (tokenizer only, not the 4B weights).

`AutoModelForCausalLM` is stock `Qwen3ForCausalLM`. We still pass `trust_remote_code=True` because `generation_config.json` sets it. Weights load in **4-bit NF4** with **float16** compute. `low_cpu_mem_usage=True` streams tensors out of the 25 GB first shard so host RAM stays under Colab's ~12–13 GB. The checkpoint config has `use_cache: false` (a training leftover); we turn the KV cache back on for short `generate()` calls.

The first run downloads about **30 GB**. That is expected.



In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "HuggingFaceBio/Carbon-X"

print("Loading tokenizer (trust_remote_code; downloads pinned Qwen3 BPE on first run)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Card example: raw DNA in ordinary text is wrapped in <dna>…</dna> and split into 6-mers.
demo_ids = tokenizer("Describe this sequence: ACGTACGATTCA")["input_ids"]
print("DNA tagging demo tokens:", tokenizer.convert_ids_to_tokens(demo_ids))

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("Loading Carbon-X in 4-bit NF4 (first run ~30 GB)...")
model = None
ATTN = None
last_error = None
for attn in ("sdpa", "eager"):
    try:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=quant,
            attn_implementation=attn,
            device_map={"": 0},
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        ).eval()
        ATTN = attn
        break
    except Exception as exc:
        last_error = exc
        print(f"{attn} 4-bit load failed:", type(exc).__name__, exc)
        if model is not None:
            del model
            model = None
        gc.collect()
        torch.cuda.empty_cache()

if model is None:
    raise RuntimeError("Could not load HuggingFaceBio/Carbon-X in 4-bit on this GPU.") from last_error

# Training leftover in config.json. Short T4 demos need the KV cache.
model.config.use_cache = True
if getattr(model, "generation_config", None) is not None:
    model.generation_config.use_cache = True

n_params = sum(p.numel() for p in model.parameters()) / 1e9
print("class:", type(model).__name__)
print(f"parameters: {n_params:.2f}B | attention: {ATTN} | quant: 4-bit nf4 / fp16 compute")
print_vram("after load")


## 4. Chat helper

The card's recipe: one user message with the sequence and the question, `apply_chat_template(..., enable_thinking=False)`, then `generate()`. Sampling defaults come from `generation_config.json` (temperature 0.6, top-p 0.95, top-k 20, `do_sample=True`) — the settings used for every reported score. `max_new_tokens` stays modest so a T4 finishes in minutes, not a quarter hour.

`skip_special_tokens=True` strips `<dna>…</dna>` tags from any DNA the model writes back.



In [ ]:
def ask(user_text, max_new_tokens=128):
    messages = [{"role": "user", "content": user_text}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True,
    ).to(DEVICE)
    prompt_len = inputs["input_ids"].shape[1]
    print(f"prompt tokens: {prompt_len} | max_new_tokens: {max_new_tokens}")
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            use_cache=True,
        )
    text = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True).strip()
    print_vram("after generate")
    return text


## 5. Demo 1 — organism group

Short public Arabidopsis *rbcL* (chloroplast RuBisCO large subunit) coding prefix. GenomeQA's organism-group task is one of Carbon-X's stronger scores on the card (93.2%). Ask for a group, not a species name — without extra annotations the model may guess an identity.

This is a research demo, not a taxonomic determination.



In [ ]:
ORGANISM_DNA = "ATGTCACCACAAACAGAGACTAAAGCAAGTGTTGGATTCAAAGCTGGTGTTAAAGATTAT"

print("=== organism group ===")
print(ORGANISM_DNA)
print()
print(
    ask(
        "Which organism group is this DNA sequence most likely from "
        "(for example mammal, plant, insect, fungus, prokaryote)? "
        "Name the group and give a short reason.\n\n"
        f"{ORGANISM_DNA}",
        max_new_tokens=128,
    )
)


## 6. Demo 2 — variant: benign vs pathogenic

The card's own HBB example (human β-globin start; the Glu6Val sickle-cell substitution is the GAG→GTG change). Prompt format from the card: **reference and alternate sequences in the same user message**, then "benign or pathogenic? Explain."

**Not clinical advice.** This is a textbook public sequence used to show the chat template. Do not treat the answer as a diagnosis or a reportable variant interpretation.



In [ ]:
# Card "How to use" example (HBB coding prefix; GAG → GTG at codon 6).
REF = "ATGGTGCATCTGACTCCTGAGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGG"
ALT = "ATGGTGCATCTGACTCCTGTGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGG"

print("=== variant (card example) ===")
print("ref:", REF)
print("alt:", ALT)
print()
print(
    ask(
        f"Reference: {REF}\n"
        f"Variant:   {ALT}\n\n"
        "Is this variant likely to be benign or pathogenic? Explain your reasoning.",
        max_new_tokens=160,
    )
)


## 7. Demo 3 — next-base prediction

NextBase is the card's sequence-recovery benchmark. Give a prefix and ask for a **small** number of following bases. Cap `max_new_tokens` — long DNA generations are slow and the card says quality drops on long designed sequence.

Print peak VRAM (`torch.cuda.max_memory_allocated`) after this last generate. The assumption in the intro is ~10–13 GB on a 15 GB T4; the number below is the measured peak for this session.



In [ ]:
PREFIX = "ATGGTGCATCTGACTCCTGAGGAGAAGTCTGCCGTT"

print("=== next-base ===")
print("prefix:", PREFIX)
print()
print(
    ask(
        "Predict the next 12 DNA bases of this sequence. "
        "Reply with the 12 bases first, then one short sentence of reasoning.\n\n"
        f"{PREFIX}",
        max_new_tokens=80,
    )
)

print()
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print_vram("end of demos")


## 8. Caveats and cleanup

**What this notebook is for.** An inference-only T4 demo of Carbon-X: hybrid DNA tokenizer, chat template with thinking off, and three short DNA questions (organism group, variant, next-base). Use it to see the prompt format and the allocated / peak VRAM.

**Not clinical advice.** Carbon-X has not been clinically validated. Do not use these outputs for medical decisions, genetic counselling, or a real patient sequence. Treat every biological claim as unverified.

**Prompt format (from the card).** Put the sequence and the question in the same user message. For variants, give reference and alternate sequences (or a reference plus position and base change). Include gene / coordinates / annotations when you have them — without context the model may invent an identity. DNA runs of ≥6 A/C/G/T are tagged automatically; you can also wrap sequences in `<dna>…</dna>`. Only A/C/G/T form 6-mers; `N` and other IUPAC codes become `<oov>` unless you filter to canonical bases first.

**Thinking.** Keep `enable_thinking=False`. The template supports thinking, but the released weights were post-trained and evaluated with it off.

**T4 / VRAM.** 4-bit NF4 + float16 compute + SDPA. Expected peak about **10–13 GB** of the T4's ~15 GB for these short prompts; the last demo cell prints `torch.cuda.max_memory_allocated`. Do not load the ~30 GB bf16 shards in float16. Do not turn FlashAttention on. The 40,960-token context window does not fit a T4 KV cache — keep prompts and `max_new_tokens` small.

**Host RAM and disk.** The first safetensors shard is 25 GB. `low_cpu_mem_usage` streams it. First download is ~30 GB plus a small Qwen3 BPE pull. If Colab's disk or host RAM dies during load, reconnect and rerun from the GPU-check cell (the Hub cache should already be warm).

**`trust_remote_code`.** `tokenizer.py` executes locally. Review it, and pin a revision if you need a frozen snapshot. The LM class is stock Qwen3.

**Not covered here.** vLLM (`vllm serve HuggingFaceBio/Carbon-X --trust-remote-code`), the card's `ask_carbon` agent tool / Inspect / Claude skill, official GGUF quants, long DNA generation, or splice-site / histone-mark questions (those GenomeQA tasks are weak on the card).

**Related weights:** [Carbon-X-GGUF](https://huggingface.co/HuggingFaceBio/Carbon-X-GGUF) if you later want llama.cpp. This notebook stays on Transformers + bitsandbytes.

**Licence.** Apache 2.0.

**Links again:** [model card](https://huggingface.co/HuggingFaceBio/Carbon-X) · [blog](https://huggingface.co/spaces/HuggingFaceBio/Carbon-X-Release) · [code](https://github.com/huggingface/carbon)

Free the GPU before you open another notebook:



In [ ]:
print_vram("before cleanup")
try:
    del model
except NameError:
    pass
try:
    del tokenizer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("Carbon-X released. Re-run the load cell if you want another prompt.")
